In [26]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from datetime import datetime
import os
import pandas as pd
import numpy as np

In [27]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CSV/CLEAN.csv')

In [28]:
#Convert data into Date time and create date filter
cleancsv['Date'] = pd.to_datetime(cleancsv['Date'])
cleancsv['Date'] = cleancsv['Date'] + pd.to_timedelta(cleancsv["Hr"], unit="h")
cleancsv.drop('Hr', axis=1, inplace=True)

"""
Use this in future if data set needs specific dates
prediction = data.loc{
    (untouched_csv['Date'] > datetime(x, x, x)) &
    (untouched_csv['Date'] < datetime(x, x, x,))
}
"""

"\nUse this in future if data set needs specific dates\nprediction = data.loc{\n    (untouched_csv['Date'] > datetime(x, x, x)) &\n    (untouched_csv['Date'] < datetime(x, x, x,))\n}\n"

In [29]:
#Create month colomn and restrict to only summer months
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8])
cleancsv = cleancsv[summer_mask].reset_index(drop=True)

In [30]:
#Prepare colomns into variables
data_main_air_temp = cleancsv['Mainland Air Temp']
data_humidity_per = cleancsv['Humidity (%)']
data_wind_direction = cleancsv['Direction (A)']
data_wind_speed = cleancsv['Wind Speed (A)']
data_gusting = cleancsv['Gusting']
data_pressure = cleancsv['Atmospheric Pressure (IN)']
data_rainfall = cleancsv['Precipitation Rate']
data_bay_temp = cleancsv['Bay Temp']
data_salinity = cleancsv['Salinity']
data_lbi_temp = cleancsv['LBI Air Temp']
data_ocean_temp = cleancsv['Ocean Temp']
data_onshore_flag = cleancsv['Onshore']
data_upwelling_flag = cleancsv['upwelling_flag']

#saves all input data into one Numpy array
dataset = np.column_stack([
    data_main_air_temp.values,
    data_humidity_per.values,
    data_wind_direction.values,
    data_wind_speed.values,
    data_gusting.values,
    data_pressure.values,
    data_rainfall.values,
    data_bay_temp.values,
    data_salinity.values,
    data_lbi_temp.values,
    data_ocean_temp.values,
    #data_onshore_flag.values,
    data_upwelling_flag.values,
])

#Save output data into variables and reshape it to be a 2d array
output_data = data_onshore_flag.values
output_data = np.array(output_data).reshape(-1, 1)

In [31]:
#Length of training data
training_data_len = int(np.ceil(len(dataset) * 0.90)) #Use 90% of training data

In [32]:
#Scaler
scaler_x= StandardScaler()
scaler_y= StandardScaler()

scaledx = scaler_x.fit_transform(dataset)
scaledy = scaler_y.fit_transform(output_data)

training_data_x = scaledx[:training_data_len] #90% of all data
training_data_y = output_data[:training_data_len] #90% of all data

X_train, y_train = [], []

In [33]:
#Sliding window over last 24 hrs
for i in range(24, training_data_len):
    X_train.append(training_data_x[i-24:i, :])
    y_train.append(training_data_y[i, 0])

#Convert lists to arrays
X_train = np.array(X_train)
y_train = np.array(y_train).reshape(-1, 1) #1D Array

In [34]:
test_x = scaledx[training_data_len-24:]
X_test = []

#rebuild window
for i in range(24, len(test_x)):
    X_test.append(test_x[i-24:i, :])

X_test = np.array(X_test)   # (samples_test, 24, n_features)
print(X_test.shape)


(218, 24, 12)


In [35]:
#Build the model
model = keras.models.Sequential()

In [36]:
#Layer Zero input_shape=(X_train.shape[1], 1)
model.add(keras.layers.Input(shape=(X_train.shape[1], X_train.shape[2])))

In [37]:
#First Layer input_shape=(X_train.shape[1], 1)
#model.add(keras.layers.LSTM(64, return_sequences=True))

In [38]:
#Second Layer
model.add(keras.layers.LSTM(32, return_sequences=False))

In [39]:
#3rd Layer (Dense)
model.add(keras.layers.Dense(32, activation="relu"))

In [40]:
#4th Layer (Dropout)
#model.add(keras.layers.Dropout(0.5))

In [41]:
#Final Output Layer (Dense)
model.add(keras.layers.Dense(1))

In [42]:
#Put all the layers together
model.summary()
model.compile(optimizer="adam",
              loss="mae",
              metrics=[keras.metrics.RootMeanSquaredError()])

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 32)             │         5,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,849 (26.75 KB)

 Trainable params: 6,849 (26.75 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
X_train,
    y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.1,
    shuffle=True
)

Epoch 1/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0472 - root_mean_squared_error: 0.0650 - val_loss: 0.0508 - val_root_mean_squared_error: 0.0619
Epoch 2/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0246 - root_mean_squared_error: 0.0322 - val_loss: 0.0406 - val_root_mean_squared_error: 0.0524
Epoch 3/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0170 - root_mean_squared_error: 0.0220 - val_loss: 0.0524 - val_root_mean_squared_error: 0.0671
Epoch 4/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0130 - root_mean_squared_error: 0.0172 - val_loss: 0.0398 - val_root_mean_squared_error: 0.0513
Epoch 5/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0117 - root_mean_squared_error: 0.0152 - val_loss: 0.0374 - val_root_mean_squared_error: 0.0501
Epoch 6/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0111 - root_mean_squared_error: 0.0143 - val_loss: 0.0354 - val_root_mean_squared_error: 0.0462
Epoch 7/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0098

In [44]:
prediction_scaled = model.predict(X_test)

# back to original units
prediction = scaler_y.inverse_transform(prediction_scaled)  


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [45]:
# rows of the original dataframe that correspond to X_test / prediction
test_index_start = training_data_len
test_index_end = training_data_len + prediction.shape[0]

test_df = cleancsv.iloc[test_index_start:test_index_end].copy()

# add predicted column
test_df["wind_pred"] = prediction.ravel()

#test_df.to_csv("CSV/predictions.csv", index=False)


In [46]:
# true wind speed for the same rows as predictions
y_test_true = cleancsv['Onshore'].iloc[test_index_start:test_index_end].values

mae = mean_absolute_error(y_test_true, prediction.ravel())
print("Test MAE (m/s):", mae)

Test MAE (m/s): 0.01150614395737648


In [47]:
print(prediction_scaled.shape)
print(prediction_scaled[:10].ravel())
print(prediction_scaled.mean(), prediction_scaled.std())

(218, 1)
[-0.00115757 -0.0100197  -0.01360327 -0.01266254 -0.01179643 -0.00420538
  0.00221937  0.00229225  0.00527647  0.00788498]
-0.0054595275 0.013329051


In [48]:

training.history['val_loss']

[0.050827398896217346,
 0.04062207415699959,
 0.05240826681256294,
 0.03979720547795296,
 0.03740984573960304,
 0.035360418260097504,
 0.03729414939880371,
 0.027172964066267014,
 0.027572060003876686,
 0.02362927235662937,
 0.026121577247977257,
 0.024273114278912544,
 0.01980380341410637,
 0.01869978941977024,
 0.01738152466714382,
 0.01647065207362175,
 0.01960729993879795,
 0.016830531880259514,
 0.014669958502054214,
 0.01546134427189827,
 0.013450488448143005,
 0.013820858672261238,
 0.012757716700434685,
 0.013086206279695034,
 0.01408038754016161,
 0.012208587490022182,
 0.01223473809659481,
 0.010235315188765526,
 0.011161400936543941,
 0.012083558365702629,
 0.011308525688946247,
 0.009607968851923943,
 0.009398085996508598,
 0.009633209556341171,
 0.011341544799506664,
 0.009383486583828926,
 0.008761007338762283,
 0.009605666622519493,
 0.008824598044157028,
 0.008754524402320385]

In [49]:
print("Wind speed mean/std:", output_data.mean(), output_data.std())
print("First 10 wind speeds:", output_data[:10].ravel())

Wind speed mean/std: 0.0 0.0
First 10 wind speeds: [0 0 0 0 0 0 0 0 0 0]


In [50]:
i0 = training_data_len   # first test index
print("Example window X[0] (first test sample):")
print(cleancsv.iloc[i0-24:i0][["Wind Speed (A)", "Mainland Air Temp", "Direction (A)"]].head(10))
print("Target at i0:", cleancsv["Wind Speed (A)"].iloc[i0])

Example window X[0] (first test sample):
      Wind Speed (A)  Mainland Air Temp  Direction (A)
1942             4.2               19.6              0
1943             4.6               18.3              0
1944             4.0               17.1             15
1945             3.9               16.1             15
1946             4.3               15.2             15
1947             3.8               14.7             15
1948             4.7               14.6             15
1949             5.7               14.1             15
1950             5.7               14.3             15
1951             6.1               14.9             15
Target at i0: 3.0
